In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

url = "https://docs.djangoproject.com/en/6.0/topics/db/models/"

response = requests.get(url)
print(response.status_code)

soup = BeautifulSoup(response.text, 'lxml')
soup.title.string

main_content = soup.find('main')

if main_content:
    headers = main_content.find_all(['h1','h2',' h3'])
    for h in headers:
        print(h.text.strip())

else:
    headers = soup.find_all('h2')
    for h in headers:
        print(h.text.strip())

In [ ]:
result = []
current_topic = "Intro"
current_content = ""
current_code = ""

all_elements = main_content.find_all(['h1', 'h2', 'h3', 'p', 'pre', 'div'])

for el in all_elements:
    if el.name in ['h1', 'h2', 'h3']:
        if current_content or current_code:
            result.append({
                'Topic': current_topic,
                'Content': current_content.strip(),
                'Code': current_code.strip()
            })
        current_content = ""
        current_code = ""
        current_topic = el.text.strip()

    elif (el.name == 'div' and 'highlight' in el.get('class', [])) or el.name == 'pre':
        current_code += el.get_text() + "\n\n"
    elif el.name == 'p':
        if not el.find_parent(['div', 'pre'], class_='highlight'):
            current_content += el.get_text().strip() + " "

result.append({
    'Topic': current_topic.replace('¶', ''),
    'Content': current_content.strip(),
    'Code': current_code.strip()
})

django_df = pd.DataFrame(result)
print(django_df.head(10))

In [79]:
django_df.to_csv('django_making_queries.csv', index=False, encoding='utf-8')

In [ ]:
test_df = pd.read_csv('django_making_queries.csv')
print(test_df.head())
print(test_df.info())

In [ ]:
print("Temat:", test_df.iloc[0]['Topic'])
print("Treść:", test_df.iloc[0]['Content'])
print("Code:", test_df.iloc[0]['Code'])